# 🧬 NLP — Named Entity Recognition (NER) with FiNER139 (using Colab)

-  📌 **Project Background**: From Biomedical to Financial NER

This notebook is the third part of a broader NER pipeline series, following two domain-specific implementations based on biomedical datasets: **BC5CDR** and **NCBI Disease Corpus**. In the previous notebooks, I built a reusable pipeline for Named Entity Recognition (NER) using Hugging Face Transformers and fine-tuned it with BioBERT for extracting disease and chemical entities from biomedical literature.

Having validated the pipeline across two different BIO tag formats (`offset_mapping`-based vs `word_ids`-based alignment), this notebook focuses on migrating the same architecture to a **new domain — Finance**, using the [FinNER-139](https://huggingface.co/datasets/nlpaueb/finer-139) dataset.

The goal is to demonstrate the **generalizability and adaptability** of the NER pipeline across domains, and to analyze:
    - What components of the pipeline remain reusable?
    - What domain-specific adjustments are required?
    - How does domain vocabulary and label distribution affect model behavior?

By switching from BioBERT to **FinBERT**, this notebook also emphasizes the importance of choosing a pre-trained model that matches the linguistic patterns of the target domain.

- 🧠 **Task Setup**

    - **Task Type**: token-classification
    - **Entity Type**: 139 entity types
    - **Dataset**:  
      🔸 Introduced by Loukas et al. in [Paper](https://arxiv.org/pdf/2203.06482v2): *"FiNER: Financial Numeric Entity Recognition for XBRL Tagging"*

      🔸 Contains 1.1M sentences with gold eXtensive Business Reporting Language (XBRL) word-level tags extracted from annual and quarterly reports of publicly-traded companies in the US.
      
      🔸 Split: 80%(train) / 10% (validation) / 10% (test)  

In [3]:
# ✅ Colab
!pip install -q transformers datasets seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.8 MB/s eta 0:00:00


In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Dataset load

In [5]:
import random
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict, set_caching_enabled
from transformers import set_seed, AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
SEED = 44
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

In [1]:
!unzip finer139_arrow_data.zip -d finer139_arrow_data

Archive:  finer139_arrow_data.zip
   creating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/finer-139-train-00001-of-00002.arrow  
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/cache-d5eaa89f91c8d424.arrow  
  inflating: finer139_arrow_data/__MACOSX/080f677a026e304c38666d759ef625d621dc8cb9/._cache-d5eaa89f91c8d424.arrow  
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/finer-139-test.arrow  
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/cache-3dc750e6e50f30cf.arrow  
  inflating: finer139_arrow_data/__MACOSX/080f677a026e304c38666d759ef625d621dc8cb9/._cache-3dc750e6e50f30cf.arrow  
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/finer-139-train-00000-of-00002.arrow  
  inflating: finer139_arrow_data/080f677a026e304c38666d759ef625d621dc8cb9/finer-139-validation.arrow  
  inflating: finer139_a

In [7]:
from datasets import Dataset, DatasetDict, concatenate_datasets

# 加载训练集两个分片
train1 = Dataset.from_file("finer139_arrow_data/data/finer-139-train-00000-of-00002.arrow")
train2 = Dataset.from_file("finer139_arrow_data/data/finer-139-train-00001-of-00002.arrow")
train = concatenate_datasets([train1, train2])

# 加载验证集和测试集
val = Dataset.from_file("finer139_arrow_data/data/finer-139-validation.arrow")
test = Dataset.from_file("finer139_arrow_data/data/finer-139-test.arrow")

# 整合为 DatasetDict
ds = DatasetDict({
    "train": train,
    "validation": val,
    "test": test
})

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


## Tokenization & Label Alignment

In [8]:
label_list = ds['train'].features['ner_tags'].feature.names
len(label_list) #139 label entities * 2 + 1 (B-label, I-label, O)

279

In [9]:
labels2id = {l:i for i, l in enumerate(label_list)}
id2labels = {i:l for i, l in enumerate(label_list)}

id2labels

{0: 'O',
 1: 'B-AccrualForEnvironmentalLossContingencies',
 2: 'B-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 3: 'I-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 4: 'B-AllocatedShareBasedCompensationExpense',
 5: 'B-AmortizationOfFinancingCosts',
 6: 'B-AmortizationOfIntangibleAssets',
 7: 'I-AmortizationOfIntangibleAssets',
 8: 'B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 9: 'I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 10: 'B-AreaOfRealEstateProperty',
 11: 'I-AreaOfRealEstateProperty',
 12: 'B-AssetImpairmentCharges',
 13: 'B-BusinessAcquisitionEquityInterestsIssuedOrIssuableNumberOfSharesIssued',
 14: 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 15: 'I-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 16: 'B-BusinessCombinationAcquisitionRelatedCosts',
 17: 'B-BusinessCombinationConsiderationTransferred1',
 18: 'B-BusinessCombinationContingentConsiderationLiabi

### Tokenizer
The domain-adaptive pretraining affects NER performance. In this case, FinBERT is suitable match for this dataset tokenizer. Careful the model is for NER task instead of semantic analysis.

In [10]:
model_name = "yiyanghkust/finbert-pretrain"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

In [11]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    labels=[]
    for i, word_ids in enumerate(tokenized_inputs.word_ids(batch_index=i) for i in range(len(example['tokens']))):
        label=[]
        prev_word=None
        for word_id in word_ids:
            if word_id is None:
                label.append(-100)
            elif word_id!= prev_word:
                label.append(example['ner_tags'][i][word_id])
            else:
                label.append(example['ner_tags'][i][word_id])
            prev_word = word_id
        labels.append(label)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

Some example exceed 512 tokens, which is not allowed in model training. `max_length=512` is added (after first training errors).

In [37]:
train_ds = ds['train'].map(tokenize_and_align_labels, batched=True, remove_columns=['id', 'tokens', 'ner_tags'])

Map:   0%|          | 0/900384 [00:00<?, ? examples/s]

### check on a sample: labels align

In [13]:
#Check what tokens are
idx=1000
sample = ds['train'][idx]
sample_t = tokenizer(sample['tokens'], truncation=True, is_split_into_words=True)
print("text:",sample)
print("---")
print("ner_tags", sample['ner_tags'])
print("tokens:",sample_t.tokens())
print("---")
print("tokenized labels:",train_ds[idx]['labels'])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


text: {'id': 1000, 'tokens': ['Additionally', ',', 'at', 'August', '31', ',', '2016', ',', 'the', 'Company', 'had', 'outstanding', 'surety', 'bonds', 'of', '$', '1.4', 'billion', 'including', 'performance', 'surety', 'bonds', 'related', 'to', 'site', 'improvements', 'at', 'various', 'projects', '(', 'including', 'certain', 'projects', 'in', 'the', 'Company', '’', 's', 'joint', 'ventures', ')', 'and', 'financial', 'surety', 'bonds', 'including', '$', '223.4', 'million', 'related', 'to', 'pending', 'litigation', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 72, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 72, 0, 0, 0, 0, 0, 0]}
---
ner_tags [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 72, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 72, 0, 0, 0, 0, 0, 0]
tokens: ['[CLS]', 'additionally', ',', 'at', 'august', '31', ',', '2016', ',', 'the', 'company', 'had', 'outstanding', 'su

In [14]:
s=[]
for i,l in enumerate(train_ds[idx]['labels']):
    if l>0:
        s.append(i)
for i in s:
    print(sample_t.tokens()[i])

1
.
4
223
.
4


In [15]:
id2labels[72]

'B-GuaranteeObligationsMaximumExposure'

Unlike biomedical NER, where entities are mostly *nouns* (diseases, chemicals), financial NER also treats quantitative phrases such as monetary values and percentages as structured entities. This reflects the domain requirement of extracting numbers for downstream financial analysis.

## Valid/Test dataset

In [38]:
valid_ds = ds['validation'].map(tokenize_and_align_labels, batched=True, remove_columns=['id', 'tokens', 'ner_tags'])
test_ds = ds['test'].map(tokenize_and_align_labels, batched=True, remove_columns=['id', 'tokens', 'ner_tags'])

Map:   0%|          | 0/112494 [00:00<?, ? examples/s]

Map:   0%|          | 0/108378 [00:00<?, ? examples/s]

The size of whole dataset is quite large however my computer resource is limited (macpro using mps here). I decide to select smaller size of dataset in training.

## Model Setup


In [17]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    label2id = labels2id,
    id2label = id2labels
)

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at yiyanghkust/finbert-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## Compute metrics

In [19]:
def compute_metrics(outputs):
    predictions, labels = outputs
    preds = np.argmax(predictions, axis=2)
    true_labels,true_preds = [],[]
    for pred, label in zip(preds, labels):
        tmp_preds, tmp_labels=[],[]
        for p_id, l_id, in zip(pred, label):
            if l_id != -100:
                tmp_preds.append(id2labels[p_id])
                tmp_labels.append(id2labels[l_id])
            true_labels.append(tmp_labels)
            true_preds.append(tmp_preds)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1_score":f1_score(true_labels, true_preds),
        "classification_report": classification_report(true_labels, true_preds)
    }

## Trainer Setup

In [45]:
train_args = TrainingArguments(output_dir='./009_2_ner_outputs',
                              eval_strategy='epoch',
                               save_strategy='epoch',
                               logging_strategy='epoch',
                               learning_rate=2e-5,
                               warmup_ratio=0.1,
                               fp16=True,
                               bf16=False,
                               per_device_train_batch_size=16,
                               per_device_eval_batch_size=16,
                               gradient_accumulation_steps=1,
                               num_train_epochs=3,
                               weight_decay=0.01,
                               load_best_model_at_end=True,
                               metric_for_best_model="f1_score",
                               save_total_limit=1,
                               logging_steps=100,
                               report_to='none'
                              )

### Callbacks

In [42]:
cbs = EarlyStoppingCallback(early_stopping_patience=2)

In [46]:
trainer = Trainer(model,
                  args=train_args,
                  train_dataset=train_ds,
                  eval_dataset=valid_ds,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  callbacks=[cbs]
                 )

In [47]:
trainer.train() #training failed in colab due to memory and runtime constraints

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.11 GiB. GPU 0 has a total capacity of 39.56 GiB of which 12.09 GiB is free. Process 63783 has 27.46 GiB memory in use. Of the allocated memory 14.54 GiB is allocated by PyTorch, and 12.41 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Experiment: Training on a Reduced Subset

Despite using Colab Pro with an A100 GPU, full-scale training on the complete FiNER139 dataset was not feasible due to memory and runtime constraints. To demonstrate the model pipeline and training flow, a reduced subset of the data was selected for experimentation:

```python
train_ds_s = train_ds.select(range(3000))
valid_ds_s = valid_ds.select(range(500))
test_ds_s = test_ds.select(range(500))


- The selected subset represents less than 1% of the full dataset (≈675K tokens total).
- Training on this small and highly imbalanced dataset — with 279 distinct entity types, many of which are extremely sparse — naturally leads to a low F1-score (<20%).
- The purpose of this run is to validate the correctness of the token classification pipeline rather than to reach optimal performance.

Please refer to the Conclusion section for the theoretical performance benchmark (~81.49% F1) based on FinBERT trained over the complete dataset, as reported by prior research.


In [26]:
trainer.train()

/Users/applewang/miniconda3/envs/fastai/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/applewang/miniconda3/envs/fastai/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/applewang/miniconda3/envs/fastai/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/applewang/miniconda3/en

TrainOutput(global_step=1875, training_loss=0.020266264088948566, metrics={'train_runtime': 621.0464, 'train_samples_per_second': 24.153, 'train_steps_per_second': 3.019, 'total_flos': 812244974632704.0, 'train_loss': 0.020266264088948566, 'epoch': 5.0})

### Evaluate valid_ds_s

In [ ]:
results = trainer.evaluate()

/Users/applewang/miniconda3/envs/fastai/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/applewang/miniconda3/envs/fastai/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
print(results["eval_classification_report"])

                                                                                                                           precision    recall  f1-score   support

                                                                                 AccrualForEnvironmentalLossContingencies       0.00      0.00      0.00      3072
                                                                                   AllocatedShareBasedCompensationExpense       0.33      0.33      0.33      9216
                                                                                           AmortizationOfIntangibleAssets       1.00      1.00      1.00      4096
                                                                                                 AreaOfRealEstateProperty       1.00      0.67      0.80      3072
                                                                   BusinessAcquisitionPercentageOfVotingInterestsAcquired       0.00      0.00      0.00      2048
                     

### Predict test_ds_s

In [ ]:
outputs = trainer.predict(test_ds_s)

In [ ]:
print(outputs.metrics["test_classification_report"])

                                                                                                                           precision    recall  f1-score   support

                                                                                   AllocatedShareBasedCompensationExpense       0.00      0.00      0.00      1152
                                                                                             AmortizationOfFinancingCosts       0.00      0.00      0.00      4320
                                                                                           AmortizationOfIntangibleAssets       0.00      0.00      0.00         0
                                                    AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount       0.00      0.00      0.00      1152
                                                                                                   AssetImpairmentCharges       0.00      0.00      0.00      1152
                     

## Conclusion & Theoretical Benchmark Reference

This notebook demonstrates a full Named Entity Recognition (NER) pipeline applied to the **FiNER139** dataset, which contains fine-grained financial entity types across ~279 label classes. 

We applied the **FinBERT** pretrained model (`yiyanghkust/finbert-pretrain`) and constructed a token-level classification framework using Hugging Face’s `Trainer` API.

📌 **Due to runtime and compute limitations on Colab**, full training over the entire dataset could not be completed. However, this experiment successfully implemented:

- Data preprocessing with token-label alignment
- Token classification modeling using FinBERT
- Metric computation using `seqeval` (entity-level F1-score)

---

### Theoretical Benchmark

In the original [FILM: Financial Language Modeling] paper by Araci et al. (2019), training FinBERT on FiNER139 yielded a benchmark **F1-score of 81.49%**.

> - 📄 Citation: Araci, D. (2019). FinBERT: Financial Sentiment Analysis with Pre-trained Language Models  
> - 🧾 GitHub: [https://github.com/deep-over/film](https://github.com/deep-over/film)  
> - 🤗 Model: `yiyanghkust/finbert-pretrain`

Given sufficient compute resources and hyperparameter tuning, **our current pipeline is theoretically expected to reach similar levels of performance**.

---

### Future Work

To further improve results and reach benchmark performance, the following directions are recommended:

- Training with full dataset and extended epochs
- Using dynamic padding and data sampling to reduce memory usage
- Applying entity label grouping to reduce class imbalance
- Exploring larger FinBERT variants or task-specific pretraining on financial corpora